# Matching and Overlap


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DESI_PATH = DATA_DIR / "DESI_halos.fits"
BULBUL_PATH = DATA_DIR / "bulbul_clusters.fits"
BALZER_PATH = DATA_DIR / "balzer_misclassified_clusters.fits"
XRAY_COMBINED_PATH = DATA_DIR / "xray_combined_catalog.csv"
DESI_OVERLAP_PATH = DATA_DIR / "desi_halos_overlap.csv"
MATCHED_PATH = DATA_DIR / "desi_xray_matched.csv"
UNMATCHED_PATH = DATA_DIR / "desi_xray_unmatched.csv"
DENOMINATOR_PATH = DATA_DIR / "desi_xray_denominator.csv"
MULTIPLE_XRAY_PATH = DATA_DIR / "desi_multiple_xray_candidates.csv"
MULTIPLE_HALO_PATH = DATA_DIR / "desi_multiple_halo_candidates.csv"


In [ ]:
import pandas as pd
from astropy.table import Table

DESI_path = DESI_PATH
bulbul_path = BULBUL_PATH
balzer_path = BALZER_PATH

def fits_to_clean_df(path, one_d_only=True, decode_bytes=True):
    table = Table.read(path, format="fits")
    if one_d_only:
        cols = [c for c in table.colnames if len(table[c].shape) <= 1]
        table = table[cols]
    df = table.to_pandas()
    if decode_bytes:
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].apply(
                    lambda x: x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else x
                )
    return df

# Bulbul
df_bulbul = fits_to_clean_df(bulbul_path)

# Balzer
df_balzer = fits_to_clean_df(balzer_path)
df_balzer["ClusterClass"] = pd.to_numeric(df_balzer["ClusterClass"], errors="coerce")
df_balzer = df_balzer[~df_balzer["ClusterClass"].isin([1, 2])].copy().reset_index(drop=True)

df_DESI = fits_to_clean_df(DESI_path)

print("DESI:", df_DESI.shape)
print("Bulbul:", df_bulbul.shape)
print("Balzer:", df_balzer.shape)


In [ ]:
import numpy as np
import pandas as pd

# -------------------------------
# 1. Select relevant Balzer columns
# -------------------------------
df_balzer_match = df_balzer[[
    "IAUName",
    "RAJ2000", "DEJ2000", "zBest",
    "ClusterClass",
    "FX300", "CRX300", "ctX300", "LX300",
    "VdispBoot", "lambdaNorm"   # velocity dispersion + richness proxy
]].copy()

# -------------------------------
# 2. Rename to match Bulbul schema
# -------------------------------
df_balzer_match = df_balzer_match.rename(columns={
    "RAJ2000": "RA_XFIT",
    "DEJ2000": "DEC_XFIT",
    "zBest": "BEST_Z"
})

# -------------------------------
# 3. Add missing columns for consistency
# -------------------------------
df_balzer_match["M500"] = np.nan
df_balzer_match["source"] = "balzer"

# -------------------------------
# 4. Add source label to Bulbul
# -------------------------------
df_bulbul = df_bulbul.copy()
df_bulbul["source"] = "bulbul"

# -------------------------------
# 5. Combine catalogs
# -------------------------------
df_xray = pd.concat([df_bulbul, df_balzer_match], ignore_index=True)

print("Combined X-ray catalog shape:", df_xray.shape)
print(df_xray.head())

# -------------------------------
# 6. Save to CSV
# -------------------------------
output_path = XRAY_COMBINED_PATH
df_xray.to_csv(output_path, index=False)

print(f"Saved combined catalog to: {output_path}")


In [ ]:
import pandas as pd
import numpy as np
from astropy.table import Table
import healpy as hp

# ==========================================
# 1. SETUP & LOAD DATA
# ==========================================
print("Loading catalogs...")

# Updated to use the new directories provided
DESI_path = DESI_PATH
xray_path = XRAY_COMBINED_PATH

# Load DESI
DESI_table = Table.read(DESI_path, hdu=1)
df_DESI = DESI_table.to_pandas()

# Filter: STRICT Centrals Only (IS_SAT == False) AND High Mass (M_HALO > 10^13)
df_DESI = df_DESI[(df_DESI['IS_SAT'] == False) & (df_DESI['M_HALO'] > 1e13)].copy().reset_index(drop=True)

# Load X-ray catalog
df_bulbul = pd.read_csv(xray_path)

# ==========================================
# 2. RESTRICTIVE REDSHIFT ENVELOPE
# ==========================================
print("Applying restrictive redshift filter...")

d_z_min, d_z_max = df_DESI['Z'].min(), df_DESI['Z'].max()
b_z_min, b_z_max = df_bulbul['BEST_Z'].min(), df_bulbul['BEST_Z'].max()

shared_z_min = max(d_z_min, b_z_min)
shared_z_max = min(d_z_max, b_z_max)

# Filter catalogs BEFORE calculating the footprint
df_DESI = df_DESI[(df_DESI['Z'] >= shared_z_min) & (df_DESI['Z'] <= shared_z_max)].copy().reset_index(drop=True)
df_bulbul = df_bulbul[(df_bulbul['BEST_Z'] >= shared_z_min) & (df_bulbul['BEST_Z'] <= shared_z_max)].copy().reset_index(drop=True)

# ==========================================
# 3. HEALPIX FOOTPRINT MAP & FILTERING
# ==========================================
print("Mapping data to HEALPix grid...")
nside = 64 # Defines resolution (~0.84 sq deg per pixel)
npix = hp.nside2npix(nside)
pix_area_sqdeg = hp.nside2pixarea(nside, degrees=True)

# Convert RA/DEC to HEALPix Theta/Phi
desi_theta = np.radians(90.0 - df_DESI['DEC'].values)
desi_phi = np.radians(df_DESI['RA'].values)
desi_pixels = hp.ang2pix(nside, desi_theta, desi_phi)

bulbul_theta = np.radians(90.0 - df_bulbul['DEC_XFIT'].values)
bulbul_phi = np.radians(df_bulbul['RA_XFIT'].values)
bulbul_pixels = hp.ang2pix(nside, bulbul_theta, bulbul_phi)

# Create global binary footprints
desi_footprint = np.zeros(npix, dtype=bool)
desi_footprint[desi_pixels] = True

bulbul_footprint = np.zeros(npix, dtype=bool)
bulbul_footprint[bulbul_pixels] = True

# Calculate the precise overlap
overlap_footprint = desi_footprint & bulbul_footprint

# Filter data strictly to objects that land in the overlapping pixels
df_DESI_overlap = df_DESI[overlap_footprint[desi_pixels]].copy().reset_index(drop=True)
df_bulbul_overlap = df_bulbul[overlap_footprint[bulbul_pixels]].copy().reset_index(drop=True)

# Calculate Area Stats
area_desi = np.sum(desi_footprint) * pix_area_sqdeg
area_bulbul = np.sum(bulbul_footprint) * pix_area_sqdeg
area_overlap = np.sum(overlap_footprint) * pix_area_sqdeg

# ==========================================
# 4. PRINT CONSOLE STATISTICS
# ==========================================
print("\n" + "="*50)
print(" HEALPIX 3D OVERLAP SUMMARY")
print("="*50)
print(f"Redshift Envelope: {shared_z_min:.4f} to {shared_z_max:.4f}")
print(f"Overlap Area: ~{area_overlap:,.0f} sq deg")
print(f"DESI Centrals in Overlap: {len(df_DESI_overlap):,}")
print(f"eROSITA Clusters in Overlap: {len(df_bulbul_overlap):,}")
print("="*50 + "\n")

# ==========================================
# 5. SAVE DESI OVERLAP DATA
# ==========================================
output_path = DESI_OVERLAP_PATH

df_DESI_overlap.to_csv(output_path, index=False)

print(f"Saved DESI overlap halos to: {output_path}")


In [ ]:
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord, search_around_sky
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.constants import G

# -------------------------------
# Paths
# -------------------------------
desi_path = DESI_OVERLAP_PATH
xray_path = XRAY_COMBINED_PATH
out_dir = DATA_DIR

# -------------------------------
# Cosmology
# -------------------------------
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# -------------------------------
# 1. Load DESI halos
# -------------------------------
df_DESI = pd.read_csv(desi_path).copy().reset_index(drop=True)

print(f"Overlap DESI halos (denominator): {len(df_DESI):,}")

# -------------------------------
# 2. Load X-ray catalog
# -------------------------------
df_xray = pd.read_csv(xray_path)

df_xray = (
    df_xray
    .dropna(subset=["RA_XFIT", "DEC_XFIT", "BEST_Z"])
    .copy()
    .reset_index(drop=True)
)

print(f"Usable X-ray sources: {len(df_xray):,}")

# -------------------------------
# 3. Compute R200c
# -------------------------------
M_halo = df_DESI["M_HALO"].values * u.Msun
z_halo = df_DESI["Z"].values

Hz = cosmo.H(z_halo).to(1 / u.s)
rho_c = 3 * Hz**2 / (8 * np.pi * G)

R200 = (
    (3 * M_halo.to(u.kg))
    / (4 * np.pi * 200 * rho_c)
) ** (1 / 3)

R200_kpc = R200.to(u.kpc).value

DA = cosmo.angular_diameter_distance(z_halo).to(u.kpc).value

theta_R200 = (
    (R200_kpc / DA) * u.rad
).to(u.arcmin)

df_DESI["R200_kpc"] = R200_kpc
df_DESI["theta_R200_arcmin"] = theta_R200.value

# -------------------------------
# 4. Coordinates
# -------------------------------
coords_DESI = SkyCoord(
    ra=df_DESI["RA"].values * u.deg,
    dec=df_DESI["DEC"].values * u.deg
)

coords_xray = SkyCoord(
    ra=df_xray["RA_XFIT"].values * u.deg,
    dec=df_xray["DEC_XFIT"].values * u.deg
)

# -------------------------------
# 5. Pre-search
# -------------------------------
idx_DESI, idx_xray, d2d, _ = search_around_sky(
    coords_DESI,
    coords_xray,
    20 * u.arcmin
)

# -------------------------------
# 6. Cuts
# -------------------------------
within_match_radius = (
    d2d < 1.5 * theta_R200[idx_DESI]
)

delta_z = np.abs(
    df_DESI["Z"].values[idx_DESI]
    - df_xray["BEST_Z"].values[idx_xray]
)

z_cut = delta_z < 0.005

mask = within_match_radius & z_cut

# -------------------------------
# 7. Candidate table
# -------------------------------
candidates = pd.DataFrame({
    "DESI_idx": idx_DESI[mask],
    "xray_idx": idx_xray[mask],
    "sep_arcmin": d2d[mask].to(u.arcmin).value,
    "delta_z": delta_z[mask],
    "R200_arcmin": theta_R200[idx_DESI[mask]].value
})

candidates["offset_r200c"] = (
    candidates["sep_arcmin"]
    / candidates["R200_arcmin"]
)

print(f"Candidate pairs: {len(candidates):,}")

# =====================================================
# TYPE 1 AMBIGUITY
# One DESI halo -> multiple X-ray candidates
# =====================================================

xray_counts_per_halo = (
    candidates
    .groupby("DESI_idx")
    .size()
    .rename("n_xray_candidates")
)

candidates = candidates.merge(
    xray_counts_per_halo,
    left_on="DESI_idx",
    right_index=True
)

candidates["multiple_xray_candidates"] = (
    candidates["n_xray_candidates"] > 1
)

type1_ambiguous = (
    candidates[
        candidates["multiple_xray_candidates"]
    ]
    .copy()
)

# -------------------------------
# 8. Resolve matches
# Keep smallest offset
# -------------------------------
best_matches = (
    candidates
    .sort_values(
        ["DESI_idx", "offset_r200c"]
    )
    .drop_duplicates(
        subset="DESI_idx",
        keep="first"
    )
    .reset_index(drop=True)
)

# =====================================================
# TYPE 2 AMBIGUITY
# Multiple DESI halos -> same X-ray source
# =====================================================

halo_counts_per_xray = (
    best_matches
    .groupby("xray_idx")
    .size()
    .rename("n_halo_candidates")
)

best_matches = best_matches.merge(
    halo_counts_per_xray,
    left_on="xray_idx",
    right_index=True
)

best_matches["multiple_halo_candidates"] = (
    best_matches["n_halo_candidates"] > 1
)

type2_ambiguous = (
    best_matches[
        best_matches["multiple_halo_candidates"]
    ]
    .copy()
)

# -------------------------------
# 9. Build matched dataset
# -------------------------------
df_DESI_pref = df_DESI.add_prefix("DESI_")
df_xray_pref = df_xray.add_prefix("xray_")

matched_full = (
    best_matches
    .merge(
        df_DESI_pref,
        left_on="DESI_idx",
        right_index=True
    )
    .merge(
        df_xray_pref,
        left_on="xray_idx",
        right_index=True
    )
)

# -------------------------------
# 10. Unmatched
# -------------------------------
matched_ids = set(best_matches["DESI_idx"])

df_unmatched = (
    df_DESI.loc[
        ~df_DESI.index.isin(matched_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# -------------------------------
# 11. Denominator
# -------------------------------
df_denominator = df_DESI.copy()

# -------------------------------
# 12. Save
# -------------------------------
matched_full.to_csv(
    MATCHED_PATH,
    index=False
)

df_unmatched.to_csv(
    UNMATCHED_PATH,
    index=False
)

df_denominator.to_csv(
    DENOMINATOR_PATH,
    index=False
)

type1_ambiguous.to_csv(
    MULTIPLE_XRAY_PATH,
    index=False
)

type2_ambiguous.to_csv(
    MULTIPLE_HALO_PATH,
    index=False
)

# -------------------------------
# Summary
# -------------------------------
print("\nSaved:")
print("- matched")
print("- unmatched")
print("- denominator")
print("- desi_multiple_xray_candidates")
print("- desi_multiple_halo_candidates")

print("\nAmbiguity summary:")
print(
    f"Halos with multiple X-ray candidates: "
    f"{type1_ambiguous['DESI_idx'].nunique():,}"
)

print(
    f"X-ray sources with multiple halo matches: "
    f"{type2_ambiguous['xray_idx'].nunique():,}"
)
